# Notebook 02 — Dataset Engineering

> **阶段**：Stage 2 Experiment · **预计时间**：40–60 分钟（CPU 可完成） · **平台**：Kaggle Notebook

数据不是「下载下来就训练」。本 Notebook 走完 检查 → 清理 → 划分 → 训练样本 的完整工程链。


# Learning Objectives

- 理解 Benchmark 数据与训练数据的关系与红线（contamination / leakage）；
- 建立确定性的 train/validation 教学子集并写出 manifest；
- 把官方 layout_dets 转换成 SFT 训练样本（图像 + 指令 + 目标 DocTags）；
- 能说明哪些转换是近似、哪些字段被跳过。


# Why This Matters

OmniDocBench 首先是 Benchmark：官方 1651 页用于评测，没有 SFT train split。如果拿测试页训练又在该集合报成绩，就是 data leakage，论文结论不可信。数据工程的意义是先划清边界，再谈训练。


# Concepts

```text
Raw OmniDocBench
      ↓
Inspect
      ↓
Clean / Validate
      ↓
Normalize
      ↓
Split（teaching: 仅 v1.5 子集）
      ↓
Training Samples（图像 + 指令 + 目标 DocTags）
```

- **Teaching Mode**：从 v1.5 子集抽取教学 train/val，产物标记 NOT for official claims；
- **Research Mode**：train / validation / official benchmark test 严格隔离，官方测试只做最终评测。


## Step 1 — 检查原始数据


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

import collections
from src import data

data_root = data.find_dataset_root()
annotations = data.load_annotations(data_root)
stats = data.build_stats(annotations)
print('总页数:', stats['pages'])
print('子集分布:', stats['subset'])

# 特殊难点属性分布（special_issue）
issues = collections.Counter()
for p in annotations:
    for s in data.page_attribute(p).get('special_issue', []) or []:
        issues[str(s)] += 1
print('special_issue Top10:', issues.most_common(10))


## Step 2 — 清理与校验

官方数据质量高，但教学工程仍要可复现的校验：ID 唯一、图像存在、文本非空、忽略标记（ignore=True 的块不参与训练目标）。


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

from src import data

ids = [data.sample_id(p) for p in annotations]
print('ID 唯一:', len(ids) == len(set(ids)))

missing = []
empty_text = 0
ignored = 0
for p in annotations:
    if not data.page_image_path(data_root, p).is_file():
        missing.append(data.sample_id(p))
    for d in p.get('layout_dets', []) or []:
        if d.get('ignore'):
            ignored += 1
        elif not d.get('text') and d.get('category_type') in ('text_block', 'title'):
            empty_text += 1
print('缺失图像:', missing[:5], '（共', len(missing), '）')
print('被 ignore 的块:', ignored, '| 无文本的正文/标题块:', empty_text)


## Step 3 — 划分教学子集

`build_teaching_split` 只从 v1.5 页面分层抽样；manifest 记录 image_id、文档类型、语言、版面与子集标记。困难子集（equation/layout/table_hard）不进入教学训练集。


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

from src import data
from src.config import project_root

N_TRAIN = 8   # TODO: GPU 环境下改为 24–100
N_VAL = 4
split = data.build_teaching_split(annotations, n_train=N_TRAIN, n_val=N_VAL, seed=42)
print('marker:', split['marker'])
print('train:', len(split['train']), '| val:', len(split['val']))

out_dir = project_root() / 'results' / 'teaching_split'
data.write_split_manifest(split, out_dir)
print('manifest 已写:', out_dir)


## Step 4 — Research Mode 的隔离原则

官方 1651 页（含困难子集）在 Phase 2 只用于 Baseline/评测（推理允许，训练禁止）。任何声称「OmniDocBench 成绩」的模型，其训练数据不得包含官方测试页。本 Notebook 的产物全部带 marker，防止混用。


## Step 5 — 生成训练样本（预览）


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

from src import data
from src.prompts import get_prompt

records = data.build_sft_records(split['train'][:2], data_root, get_prompt('v0'))
for r in records:
    print('image_id:', r['image_id'])
    print('instruction:', r['instruction'][:80])
    print('target_doctags（前 400 字符）:')
    print(r['target_doctags'][:400])
    print('api_note:', r['api_note'], '| 跳过类别:', r['skipped_categories'])
    print('---')


# What You Should Observe

- 训练目标 DocTags 是**派生转换**（layout_dets → DoclingDocument → DocTags），不是官方原生 SFT 数据；表格/公式结构被跳过时，文字仍按顺序保留；
- manifest 与 marker 让任何后续使用者都能一眼看出数据边界。


# Research Checkpoint

> **为什么不能拿 Benchmark test set 训练后再报告成绩？** 从 data leakage、指标高估、以及论文复现性三个角度说明。

**TODO：** 答案写入 `results/nb02/research_checkpoint.md`。


# Exercises

1. **TODO：** 检查 val 与 train 是否存在同一 image_id 重叠，写断言验证；
2. **TODO：** 挑一个被 `page_to_doctags` 跳过的 table 页，说明为什么表格结构转换是难点，并提出一种保守的转换方案；
3. **TODO：** 把 N_TRAIN 改小/改大各跑一次，观察 manifest 的文档类型分布变化（分层抽样是否保持均衡）。


# Takeaways

- 数据边界先于训练：teaching subset 与官方 benchmark 隔离是第一红线；
- 派生训练目标必须如实标注近似性；
- manifest 是可复现数据工程的产物，不是可有可无的记录。

**下一步**：[Notebook 03](03_Prompt_Engineering.ipynb) — 把 Prompt 当作实验变量。
